# Big Data Platforms — Lecture 6 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC EXAM, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 17.9.2026

Lectures 3–5 were about storage — how to keep bytes safe and fast. Lecture 6
turns to a different, harder problem: what happens when the *state itself* has
to be shared across multiple machines that can't always talk to each other? It
starts from ordinary database ACID guarantees, shows why they get genuinely hard
to keep once a database is distributed (Brewer's CAP theorem), works through the
practical CA/CP/AP trade-off with real systems as examples, then goes one level
deeper into *why* keeping distributed nodes in agreement is fundamentally
hard (the FLP impossibility result), and how practical systems get around that
anyway (Paxos, Raft) and package it up so ordinary application code never has to
touch the hard part directly (Chubby, ZooKeeper, Raft-based systems like etcd).


## 1. ACID — what a single database promises

Before asking what breaks when a database gets distributed, it's worth being
precise about what a traditional, centralized database guarantees in the first
place. These are the **ACID** properties:

- **Atomicity** — a transaction is "all or nothing." If any part of it fails,
  the whole transaction fails and the database is left exactly as it was
  before the transaction started — no partial changes linger around.
- **Consistency** (the database-theory sense, not Brewer's sense below —
  worth flagging early since the word gets reused with a different meaning in
  a few slides) — every transaction moves the database from one *consistent*
  state to another, meaning it never leaves data violating whatever integrity
  rules the schema defines.
- **Isolation** — transactions can't interfere with each other. Each one
  behaves as though it were the only thing running against the database, even
  when many are actually running concurrently.
- **Durability** — once a transaction is committed, it stays committed,
  surviving crashes, power loss, whatever comes next.

These four properties are what makes programming against a single database
feel, in a sense, "safe by default." The rest of this lecture is essentially the
story of how much of that safety survives once the database has to span
multiple machines.


## 2. Distributed databases and Brewer's three properties

Cloud computing forces the issue: to scale to a large number of users sharing
common state, the database has to be *distributed* across many machines. Eric
Brewer popularized three general properties relevant to any distributed system —
note that the first one, "Consistency," is defined slightly differently here
than the ACID sense above:

- **Consistency** (Brewer's sense) — every node has a consistent view of the
  distributed database's contents. All nodes agree on what the data currently
  is.
- **Availability** — every request eventually gets *some* response — success
  or failure — rather than hanging forever with no answer at all.
- **Partition tolerance** — the system keeps operating even when arbitrary
  messages between nodes get lost (a "network partition" — some nodes can't
  reach others).


## 3. Brewer's CAP theorem

At a PODC 2000 invited talk, Eric Brewer conjectured something that sounds
almost too strong to be true: **no distributed system can simultaneously
guarantee all three of Consistency, Availability, and Partition tolerance.**
This wasn't just a hunch that stuck around — it was formally proven as a real
theorem by Seth Gilbert and Nancy Lynch (*Brewer's conjecture and the
feasibility of consistent, available, partition-tolerant web services*, SIGACT
News 33(2):51–59, 2002).

Since you provably can't have all three, every distributed system design is
really a choice of **which two** to keep, at the expense of the third:

- **CA** — Consistent and Available, but *not* Partition tolerant. This is
  really just describing a **non-distributed, centralized** database — there's
  no partition to tolerate because there's only ever one copy of the truth.
- **CP** — Consistent and Partition tolerant, but *not* Available. When the
  network splits, the system would rather refuse to answer than risk giving an
  answer that might be wrong.
- **AP** — Available and Partition tolerant, but *not* Consistent. When the
  network splits, the system keeps answering requests on both sides of the
  split — but the two sides may now disagree with each other about the data.


It's worth being honest about what "choosing CA" actually means in
practice: you're not really solving the trilemma, you're opting out of it by
refusing to be a distributed system in the first place. CP and AP are the two
*genuinely distributed* choices, and the entire rest of this lecture is about
what each of those two choices costs you, and how real systems live with that
cost.


## 4. Example CA systems

CA effectively means "not really distributed" — a single authoritative copy of
the data, with no partition problem to worry about because there's nothing to
partition:

- Single-site (non-distributed) databases, generically.
- **LDAP** — Lightweight Directory Access Protocol, commonly used for user
  authentication.
- **NFS** — Network File System.
- Centralized version control systems, e.g. **SVN**.
- The **HDFS NameNode** — a callback to lectures 3–4. Its single-NameNode design
  (before Federation/HA) is precisely a CA system: one authoritative copy of the
  filesystem metadata, simple to reason about, but with no partition tolerance
  built in at all.


## 5. Example CP systems

CP systems keep every node's view of the data consistent, and remain tolerant of
network partitions — but the price is that they'll refuse to serve some
requests (particularly writes) during a partition, rather than risk
inconsistency.

- Distributed databases such as **Google Bigtable** and **Apache HBase**.
- Distributed coordination systems such as **Google Chubby** and
  **Apache ZooKeeper** — both of which get their own dedicated sections later in
  this lecture.

The mechanism underneath most of these: a **master copy** location for
modifications, combined with either pessimistic locking or a **majority
(quorum) algorithm** — Leslie Lamport's **Paxos** is the canonical example, and
gets its own section below too.


## 6. Example AP systems

AP systems stay available and partition-tolerant, but nodes can end up
disagreeing with each other about the data, at least temporarily:

- Web caching, generically.
- **DNS** — the Domain Name System.
- **Git** — a distributed version control system. Two people can commit
  divergent history on their own machines while fully disconnected from each
  other; reconciling that later is exactly the AP trade-off in miniature.
- Filesystems built to support disconnected operation, such as **AFS** and
  **Coda**.
- "**Eventually consistent**" datastores — **Amazon Dynamo**, **Apache
  Cassandra**.

The mechanisms that make this workable: cache **expiration times** and
**leases**, and — where the data model allows it — intelligent automatic
**merge** logic for reconciling independent updates once connectivity is
restored.


## 7. Trade-offs, side by side

### CA systems
- Limited scalability — a reasonable choice only when expected system load is
  genuinely low.
- Easiest to implement, and easiest to program against — there's no partition
  behaviour to reason about at all.
- High latency if clients aren't geographically close to the single copy, since
  there's nowhere closer for them to talk to.

### CP systems
- Good scalability.
- Relatively easy to program against — reads and writes still behave in a
  fairly intuitive, single-copy-like way most of the time.
- Data consistency is achieved, and it scales — but updates carry high latency,
  since a write typically has to be acknowledged by a majority of nodes before
  it's considered committed.
- Quorum algorithms like Paxos keep the system *available* as long as a
  majority of components can still reach each other.
- But once a network partition actually happens, a CP system will eventually
  become **visibly unavailable for updates** on whichever side of the partition
  doesn't hold the majority.

### AP systems
- Extremely good scalability — this is the trade-off's biggest selling point.
- **Very difficult to program against.** There is no single general-purpose
  algorithm for merging inconsistent updates. The analogy the lecture draws is
  distributed version control: there will always be some conflicting changes
  that genuinely can't both be applied, and no universal rule for which one
  should "win" — that decision is inherently application-specific.
- Data consistency is explicitly sacrificed — though this is often tolerable in
  practice (web page caching being the obvious example: a slightly stale cached
  page is rarely a real problem).
- Low-latency updates are possible precisely because a client can always update
  its *closest* copy immediately, letting the system propagate that change to
  other copies in the background afterward.
- The cost of that low latency: the system **will eventually show visibly
  inconsistent data** to users, right after a network partition, until
  propagation catches up.

### Why choose AP at all?

A large share of core Internet infrastructure is AP by design — web caching and
DNS both being prime examples. AP tends to work best for data that changes
rarely, is often simpler to implement at scale than a CP system (sometimes as
simple as "cache with a timeout"), and is the only real option when an
application genuinely needs low latency above all else. The catch is that it
needs application-specific code to actually reconcile inconsistent updates —
there's no generic mechanism that solves this for you, unlike CP's fairly
uniform quorum-based approach.


## 8. Real systems mix CA, CP, and AP within one application

Almost nothing running at real scale picks just one of the three and uses it
everywhere. The interesting engineering happens in *how* a system splits
different pieces of its state across the three categories.

### Amazon's web store
- **Shopping basket: AP.** Always available for updates, even mid-partition. If
  two divergent basket copies show up after a partition heals, the merge policy
  is simple and safe: take the **union** of both baskets' contents — worst case,
  you end up with an item you didn't strictly want to add twice, not a lost
  item or a broken order.
- **Billing, and the master copy of inventory: CP.** These need a single
  authoritative, consistent answer — you can't let two different systems both
  believe they've sold the last unit of something.
- Inconsistencies between the AP basket and the CP inventory/billing systems
  surface at **order confirmation time**, where an item might need to be
  backordered or cancelled if the two disagreed. And the added latency of the
  CP system for billing is often conveniently hidden behind another naturally
  slow step anyway — sending the confirmation email, for instance.

### Google Gmail
- **Marking a message read: AP.** This needs to be available at all times, and
  the merge rule is again trivially simple — union of "read" flags across any
  divergent copies.
- **Sending an email: CP.** An email either gets sent or it doesn't, and the
  user needs a definite acknowledgment either way — there's no safe "maybe sent,
  we'll reconcile later" answer for this particular action.

### A Hadoop-based web application
- **HDFS NameNode: CA.** A centralized filesystem-namespace database — simple to
  implement, at the cost of being a single point of failure (echoing lecture 3's
  discussion).
- **HBase: CP.** A distributed database that will refuse modifications under a
  network partition, coordinated through Apache ZooKeeper.
- **An optional web caching layer in front of both: AP.** Purely there to cut
  down request load on the CP/CA systems behind it.

### The general pattern

Put together, a clear rule of thumb emerges: **AP is usually used for the
user-facing interaction layer**, because it's cheaper to run at scale and gives
lower latency, at the cost of needing bespoke merge logic. **CP holds the
authoritative "master copy"** of anything that genuinely can't tolerate
disagreement, accepting the latency that comes with quorum-based writes — and
that latency needs to be hidden from the user wherever possible. **CA still
shows up too**, usually because it's the simplest thing to implement, provided
the architecture is careful not to let that single centralized piece become the
whole system's bottleneck.


## 9. Two-phase commit

Once a transaction needs to span multiple database servers, *something* has to
coordinate whether it commits everywhere or rolls back everywhere — you can't
have half the servers commit and the other half roll back the same logical
transaction. The most widely used protocol for this is **two-phase commit**
(2PC): <http://en.wikipedia.org/wiki/Two-phase_commit_protocol>.

Two roles are involved: a **coordinator**, and the database servers involved in
the transaction, called **cohorts**. The protocol has exactly two phases:

1. **Voting phase** — the coordinator asks every cohort to *prepare* for the
   transaction (essentially: "can you commit this if asked?").
2. **Completion phase** — if every cohort voted yes, the coordinator tells all
   of them to **commit**; if even one voted no (or didn't respond), it tells
   all of them to **roll back**.


In [1]:
# A minimal simulation of two-phase commit, purely to make the two-phase
# structure concrete: a voting phase, then a completion phase that only
# commits if every cohort voted yes.

class Cohort:
    def __init__(self, name, will_vote_yes=True):
        self.name = name
        self.will_vote_yes = will_vote_yes
        self.state = "idle"

    def vote(self):
        # Phase 1: "can you commit this transaction if asked?"
        return self.will_vote_yes

    def complete(self, decision):
        # Phase 2: coordinator's final decision is applied.
        self.state = decision
        return self.state


def two_phase_commit(cohorts):
    # Phase 1: voting
    votes = {c.name: c.vote() for c in cohorts}
    all_yes = all(votes.values())

    # Phase 2: completion
    decision = "commit" if all_yes else "rollback"
    for c in cohorts:
        c.complete(decision)

    return votes, decision


# Scenario 1: every cohort is healthy and votes yes -> transaction commits
cohorts = [Cohort("db-a"), Cohort("db-b"), Cohort("db-c")]
votes, decision = two_phase_commit(cohorts)
print("Scenario 1 (all healthy):", votes, "->", decision)
print("  final states:", {c.name: c.state for c in cohorts})

# Scenario 2: one cohort can't commit -> the whole transaction rolls back,
# even though the other two cohorts were perfectly willing to commit
cohorts = [Cohort("db-a"), Cohort("db-b", will_vote_yes=False), Cohort("db-c")]
votes, decision = two_phase_commit(cohorts)
print("\nScenario 2 (one cohort votes no):", votes, "->", decision)
print("  final states:", {c.name: c.state for c in cohorts})


Scenario 1 (all healthy): {'db-a': True, 'db-b': True, 'db-c': True} -> commit
  final states: {'db-a': 'commit', 'db-b': 'commit', 'db-c': 'commit'}

Scenario 2 (one cohort votes no): {'db-a': True, 'db-b': False, 'db-c': True} -> rollback
  final states: {'db-a': 'rollback', 'db-b': 'rollback', 'db-c': 'rollback'}


### What happens when something crashes

If a **cohort** crashes mid-transaction, the system can automatically roll back
whatever was in progress for that cohort — this case is handled cleanly.

The genuinely dangerous case is the **coordinator** crashing partway through the
completion phase, permanently. Some cohorts may have already been told to
commit or roll back; others may be left waiting indefinitely for a message that
will never arrive, with no way to know on their own whether the "right" answer
was commit or rollback. This isn't a bug that a smarter implementation can patch
around — it's structural: **two-phase commit cannot be made fault-tolerant
without changing the protocol itself.** A cohort stuck in that state can, in the
worst case, **block forever.**

This single fact — that the most common real-world transaction coordination
protocol is not fault tolerant against coordinator failure — is exactly the
motivation for everything that follows in this lecture: consensus, FLP, Paxos,
and the distributed coordination services built on top of them.


## 10. The asynchronous consensus problem

Strip away the database-specific framing and 2PC's coordinator-failure problem
is really an instance of a much more general, classic problem in distributed
systems: **consensus** — getting a set of processes to agree on a single value,
even when some of them might fail.

The formal setup:

- There are *N* processes.
- Each process *i* starts with an **initial vote** *vᵢ ∈ {0, 1}*.
- The network is **asynchronous** — there's no upper bound on how long a
  message can take to arrive — but **reliable**: every message that's sent is
  *eventually* delivered, just with no guaranteed timing. Critically, there's
  also **no shared clock** the processes can rely on.
- **At most one process fails**, and it fails only by *halting* — going
  permanently and silently unresponsive (this is the simplest possible failure
  model; it doesn't even allow for a process sending wrong or malicious
  messages, just going quiet).
- The protocol must **terminate in finite time**, deciding either 0 or 1 — and
  whatever it decides must actually be *someone's* initial vote (otherwise the
  problem is trivial: you could just always decide 0 regardless of what anyone
  voted).


## 11. The FLP theorem — consensus is provably impossible in general

Given a problem framed that carefully, it's natural to ask: is there an
algorithm that solves it? Fischer, Lynch, and Paterson answered this in one of
the most famous impossibility results in distributed computing: **"Impossibility
of Distributed Consensus with One Faulty Process"** (Fischer, Lynch, Paterson,
*J. ACM* 32(2): 374–382, 1985).

**The result: no algorithm can solve the asynchronous consensus problem, full
stop** — not "no algorithm we've found yet," but a genuine impossibility proof.

**The key intuition behind why:** in a fully asynchronous network — no bound on
message delay — there is *no way to distinguish* a process that has actually
failed from a process that is simply extremely slow, with all of its messages
delayed. Any protocol that tries to make progress by assuming a silent process
has failed can always turn out to be wrong; that "failed" process might just be
about to deliver a very late message that changes everything.

The direct consequence: any correct, fault-tolerant consensus protocol has some
runs that genuinely **never terminate** — runs where the protocol keeps waiting
on a process that it can never be sure has actually failed. Interestingly, the
moment you *do* add an upper bound on message transmission delay, the whole
problem becomes solvable again — FLP is specifically a statement about the
asynchronous case.

### What the non-terminating runs actually look like

Digging into the proof a bit further: FLP shows any sound fault-tolerant
consensus protocol has *some* run that never reaches a decision. In practice,
though, these non-terminating runs consist of an infinite sequence of situations
where different processes keep getting suspected of having failed (because
their messages are delayed), one after another, forever — not one process being
permanently and obviously stuck.

### Why this doesn't actually stop people from building consensus systems

Here's the resolution that makes the rest of this lecture possible: in any
**real** computer network, the probability of hitting one of these
never-ending "everyone keeps suspecting everyone else" runs is vanishingly
small. So while it's mathematically true that no algorithm can *guarantee*
termination, a well-designed practical algorithm can still solve consensus
**with extremely high probability**, and terminate quickly essentially every
time it actually runs in the real world. FLP tells you the theoretical ceiling;
it doesn't tell you the algorithm will actually hang in practice.


## 12. Paxos

**Paxos**, from Leslie Lamport's paper *The Part-Time Parliament* (ACM Trans.
Comput. Syst. 16(2): 133–169, 1998), is exactly that kind of practically-sound
algorithm: designed for the asynchronous consensus problem, living within the
constraints FLP proved are unavoidable.

**The guarantee Paxos actually makes:** *if* Paxos terminates, its output is
guaranteed to be a correct consensus outcome. It does **not** claim to always
terminate — FLP still applies, that's not negotiable — but in practice it
terminates with very high probability.

### The mechanics

- Paxos uses **2f + 1** servers to tolerate **f** concurrent failures while
  still making progress. Tolerating two simultaneous failures needs **five**
  servers, not three or four.
- The key intuition behind that formula: if every modification is saved on at
  least **f + 1** servers — a **majority**, or **quorum** — then even if *f* of
  those servers fail, at least one server holding that modification is
  guaranteed to still be alive.
- For efficiency, Paxos implementations commonly route all modifications through
  a **centralized leader**, rather than every server coordinating with every
  other server on every write.
- If the current leader is suspected to have failed, a new leader can (with
  high probability) be elected quickly, and modifications simply get redirected
  to the new leader going forward.


In [2]:
# Illustrating the 2f+1 quorum requirement directly.

def servers_needed(f):
    """Minimum servers needed to tolerate f concurrent failures under Paxos."""
    return 2 * f + 1

for f in range(0, 5):
    n = servers_needed(f)
    majority = f + 1
    print(f"To tolerate f={f} concurrent failures: need {n} servers "
          f"(a majority/quorum of {majority} must persist each write)")


To tolerate f=0 concurrent failures: need 1 servers (a majority/quorum of 1 must persist each write)
To tolerate f=1 concurrent failures: need 3 servers (a majority/quorum of 2 must persist each write)
To tolerate f=2 concurrent failures: need 5 servers (a majority/quorum of 3 must persist each write)
To tolerate f=3 concurrent failures: need 7 servers (a majority/quorum of 4 must persist each write)
To tolerate f=4 concurrent failures: need 9 servers (a majority/quorum of 5 must persist each write)


## 13. Distributed coordination systems — modularizing the hard part

The lecture's next move is a genuinely important piece of engineering advice, not
just a factual claim: **consensus-under-failure protocols are extremely subtle,
and normal application code should not be trying to implement them from
scratch.** The standard solution is to push all of that subtlety into a single,
carefully-built, carefully-tested **distributed coordination system**, and have
every application that needs consensus talk to *that* instead of reinventing it.


## 14. Google Chubby

**Google Chubby** (Michael Burrows, *The Chubby Lock Service for Loosely-Coupled
Distributed Systems*, OSDI 2006: 335–350) is one of the best-known systems built
directly on Paxos.

Chubby's job inside Google's infrastructure: storing configuration parameters
for essentially all other Google services, electing leaders, providing locking,
maintaining pools of "which servers are currently up," and mapping names to IP
addresses. That's a lot of very different-sounding jobs, and the common thread
is that every one of them ultimately reduces to "get a set of distributed
processes to agree on something" — exactly the consensus problem, wearing
different clothes each time.

**A structural limitation worth flagging:** because practical Paxos
implementations route all writes through a single leader, that leader's own
write throughput becomes a hard ceiling on the whole system's write throughput.
This isn't a Chubby-specific quirk — it recurs below with ZooKeeper too.


## 15. Apache ZooKeeper

**Apache ZooKeeper** is the open-source counterpart to Chubby, though it's
built on a *different* protocol — not Paxos itself, but one that shares many of
the same ideas. The underlying protocol is **Zab** (Flavio Junqueira and
Benjamin Reed, *Brief Announcement: Zab: A Practical Totally Ordered Broadcast
Protocol*, DISC 2009: 362–363).

(Figures referenced in this section of the lecture come from
<http://zookeeper.apache.org/doc/r3.4.0/zookeeperOver.html>.)

### The data model

ZooKeeper presents clients with something like a small, fault-tolerant
**filesystem**, complete with access-control management. Each node in that
filesystem is called a **znode** — a combined file-and-directory concept:
a znode can have child znodes, *and* up to **1MB** of data attached directly to
it. Reads and writes of a znode's data are **atomic** — the full up-to-1MB
payload is read or written as a single indivisible unit, so there's no risk of
observing a half-written value.


###ZooKeeper namespace (hierarchical znode tree)
![ZooKeeper namespace (hierarchical znode tree)](images/zookeeper_Namespace.png)


### Ephemeral znodes and watches

Beyond ordinary persistent znodes, ZooKeeper supports **ephemeral znodes**: a
znode that exists for exactly as long as the client that created it keeps
replying to ZooKeeper's keepalive messages in a timely manner. The moment a
client stops responding in time, ZooKeeper assumes it has failed, and its
ephemeral znodes disappear. This gives applications a clean, built-in way to
**detect failed nodes** — register an ephemeral znode representing "I am alive
and doing X," and any other client can tell you've died simply by noticing that
znode vanished.

ZooKeeper also lets clients register **watches** — a subscription to be
notified the moment a particular znode changes. This means an application can
learn about a server joining or leaving the pool *reactively*, the instant it
happens, rather than having to continuously poll ZooKeeper and burn resources
checking for a change that usually hasn't happened yet.


### How the ZooKeeper service itself works

- Each client connects to exactly **one** ZooKeeper server.
- Every **write** request gets forwarded through a single **Leader** node.
- The Leader puts all writes into a single **total order**, and forwards that
  ordered stream to the **Follower** nodes.
- A write is only acknowledged back to the client once a **majority** of
  servers have durably persisted it — the same quorum idea Paxos relies on.

### ZooKeeper's read/write characteristics

- **Reads are served locally**, from the cache of whichever server the client
  happens to be connected to — which means a read can return data that's
  slightly out of date, by up to roughly tens of seconds in the worst case.
- A `sync()` call exists specifically for when that staleness is unacceptable:
  it forces the connected server to catch up with the Leader before answering,
  trading a bit of latency for a fully up-to-date read.
- Because a write is only acked once a **majority** of servers have it durably
  persisted, if a *minority* of servers fail, there's always guaranteed to be at
  least one surviving server holding the most up-to-date committed data.
- If the Leader itself fails, ZooKeeper automatically selects a new one.


### ZooKeeper components (Leader / Follower architecture)
![ZooKeeper components (Leader / Follower architecture)](images/zookeeper_components.png)


### ZooKeeper performance — the read/write asymmetry

One of the more counter-intuitive but important facts in this lecture: adding
**more servers** to a ZooKeeper ensemble helps performance **only** for
read-heavy workloads. That makes sense once you recall the mechanics above:
reads are served locally by whichever server a client is connected to, so more
servers means more places to spread read load across.

**Writes don't get that benefit at all**, because every single write has to be
funneled through the one Leader and then acknowledged by a majority of
servers — adding more followers doesn't parallelize that, it just means the
quorum has to include more machines. The practical rule of thumb the lecture
gives: **if more than roughly 30% of operations are writes, the minimum
three-server configuration actually performs best** — extra servers beyond
that minimum quorum size just add coordination overhead for no throughput
benefit.


###ZooKeeper performance vs. number of servers (read vs write heavy workloads)
![ZooKeeper performance vs. number of servers (read vs write heavy workloads)](images/zookeeper_performance.png)


In [3]:
# A toy illustration of the effect described above: modelling read
# throughput as scaling with server count (served locally from each
# server's cache), and write throughput as *shrinking* as the ensemble
# grows, since every write needs acks from a larger quorum before it's
# considered committed. Purely qualitative -- not a real performance model,
# just enough to show why a 3-server ensemble can beat a 7-server one once
# writes dominate.

def approx_zookeeper_throughput(num_servers, write_fraction,
                                 per_server_read_capacity=10, write_capacity_base=1000):
    read_fraction = 1 - write_fraction
    # Reads scale with server count (served locally from each server's cache)
    read_capacity = num_servers * per_server_read_capacity
    # Writes get slower as the quorum grows: more followers to hear back from
    # before the Leader can ack the write, so total write capacity shrinks.
    write_capacity = write_capacity_base / num_servers

    # Rough blended throughput for the given read/write mix
    return read_fraction * read_capacity + write_fraction * write_capacity


for write_fraction in [0.05, 0.30, 0.60]:
    print(f"\nWrite fraction = {write_fraction:.0%}")
    best = None
    for n in [3, 5, 7]:
        t = approx_zookeeper_throughput(n, write_fraction)
        print(f"  {n} servers -> approx blended throughput: {t:,.0f} ops/sec")
        if best is None or t > best[1]:
            best = (n, t)
    print(f"  -> best ensemble size at this write fraction: {best[0]} servers")



Write fraction = 5%
  3 servers -> approx blended throughput: 45 ops/sec
  5 servers -> approx blended throughput: 58 ops/sec
  7 servers -> approx blended throughput: 74 ops/sec
  -> best ensemble size at this write fraction: 7 servers

Write fraction = 30%
  3 servers -> approx blended throughput: 121 ops/sec
  5 servers -> approx blended throughput: 95 ops/sec
  7 servers -> approx blended throughput: 92 ops/sec
  -> best ensemble size at this write fraction: 3 servers

Write fraction = 60%
  3 servers -> approx blended throughput: 212 ops/sec
  5 servers -> approx blended throughput: 140 ops/sec
  7 servers -> approx blended throughput: 114 ops/sec
  -> best ensemble size at this write fraction: 3 servers


Even in this deliberately simplified model, the pattern the lecture
describes shows up: at a low write fraction, adding servers keeps helping,
since most requests are reads that scale with server count. As the write
fraction climbs, the shrinking per-write throughput of a larger quorum starts
to outweigh the read-scaling benefit, and the smallest viable ensemble (three
servers) becomes the better choice — which is exactly the lecture's practical
guidance once writes make up a substantial share of the workload.


## 16. Raft — a more understandable alternative

**Raft** (Diego Ongaro and John Ousterhout, *In Search of an Understandable
Consensus Algorithm*, USENIX ATC 2014: 305–319) is a widely-used alternative to
both Paxos and Zab. Its explicit design goal, stated right in the paper's
title, was **understandability** — Paxos in particular has a well-earned
reputation for being conceptually simple to state but genuinely difficult to
implement correctly, and Raft was designed specifically to close that gap
between "sounds simple" and "is simple to build correctly."

Raft is now implemented across many different systems and languages — see
<https://raft.github.io/> for the ecosystem. A concrete, widely-deployed
example: **etcd**, a Raft-based key-value store that serves as the fault-tolerant
core of the **Kubernetes** container-management system. Every time a Kubernetes
cluster's control plane needs to agree on cluster state, it's ultimately
leaning on Raft consensus running inside etcd underneath it.


## 17. Summary: why distributed coordination services exist at all

Pulling the whole lecture together into the point it's actually building toward:

Keeping a consistent, shared view of global state across a set of machines that
can fail and can't always talk to each other is a genuinely **subtle** problem —
subtle enough that FLP proves it's impossible to solve with a hard guarantee,
and subtle enough that even the most widely used practical protocol for
distributed transactions (two-phase commit) is provably not fault tolerant
against coordinator failure.

The right engineering response isn't to have every application team reinvent
this from scratch — it's to **implement these subtle algorithms exactly once**,
inside a dedicated, heavily-tested distributed coordination service (Chubby,
ZooKeeper, or a Raft-based system), and have every application built on top of
the cloud simply *use* that service rather than reimplementing consensus itself.

From an application's point of view, the coordination service becomes the place
you store things like: global configuration data, global locks, where the
current master server lives, which slave servers are currently alive, and so
on — exactly the kind of small, critical, must-agree-on-it state that the CAP
theorem, FLP, and the two-phase-commit failure mode all warn you not to handle
casually. By pushing all of that into one modular, well-understood service,
every other piece of application code gets to stay dramatically simpler — free
to assume "the coordination layer already solved this" rather than solving it
itself, badly, one more time.
